# Softmax + Cross-Entropy

Wiki reference for [softmax cross-entropy](https://ml-viz-ruby.vercel.app/wiki/softmax-cross-entropy).

**The idea in one sentence.** Softmax turns logits into a distribution and cross-entropy scores
$-\log(\text{prob of the true class})$ — and their gradient w.r.t. the logits is the beautifully
simple **$\hat p - \text{one-hot}(y)$**; the one practical catch is that softmax must subtract the
max before exponentiating to avoid overflow.

We implement softmax, cross-entropy, and their gradient from scratch, **validate the gradient
identity and numerical stability**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.style.use('dark_background')

## From-scratch implementation

In [ ]:
def softmax(z):
    """Numerically stable softmax (subtract max before exp)."""
    e = np.exp(z - z.max())
    return e / e.sum()

def cross_entropy_loss(z, y):
    """Cross-entropy loss for true class y."""
    return -np.log(softmax(z)[y] + 1e-12)

def softmax_ce_grad(z, y):
    """Exact gradient: dL/dz_k = p_hat_k - 1[k==y]."""
    grad = softmax(z).copy()
    grad[y] -= 1
    return grad

## Reproduce the 3-class worked example

In [ ]:
z = np.array([2.0, 1.0, -1.0])
y = 0  # true class

p_hat = softmax(z)
loss  = cross_entropy_loss(z, y)
grad  = softmax_ce_grad(z, y)

print(f"Logits z:       {z}")
print(f"Softmax p̂:     {p_hat.round(4)}")
print(f"Loss:           {loss:.4f}   (= -log({p_hat[y]:.4f}))")
print(f"Exact gradient: {grad.round(4)}")
print(f"  correct class k=0: {grad[0]:.4f}  = {p_hat[0]:.4f} - 1")
print(f"  wrong   class k=1: {grad[1]:.4f}  = {p_hat[1]:.4f}")
print(f"  wrong   class k=2: {grad[2]:.4f}  = {p_hat[2]:.4f}")
print(f"  gradient sums to:  {grad.sum():.6f}  (should be 0)")

### Validate: the gradient is $\hat p - \text{one-hot}(y)$

The gradient of softmax-cross-entropy w.r.t. the logits collapses to the predicted probabilities
minus the one-hot true label — and the loss is $-\log$ of the true-class probability. We confirm
both.

In [ ]:
onehot = np.eye(len(z))[y]
print(f'grad = {grad.round(4)};  p_hat - one_hot = {(p_hat - onehot).round(4)}')
assert np.allclose(grad, p_hat - onehot), 'the softmax-CE gradient is p_hat - one_hot(y)'
assert np.isclose(loss, -np.log(p_hat[y]), atol=1e-6), 'CE loss = -log(probability of the true class)'
print('\n✅ the elegant gradient: predicted distribution minus the target')

## Finite-difference verification

In [ ]:
eps = 1e-5
fd = np.zeros(len(z))
for i in range(len(z)):
    z_p, z_m = z.copy(), z.copy()
    z_p[i] += eps; z_m[i] -= eps
    fd[i] = (cross_entropy_loss(z_p, y) - cross_entropy_loss(z_m, y)) / (2 * eps)

print(f"Finite-diff grad: {fd.round(6)}")
print(f"Exact grad:       {grad.round(6)}")
print(f"Max abs error:    {np.abs(grad - fd).max():.2e}")
print(f"Gradients match:  {np.allclose(grad, fd, atol=1e-7)}")

### Validate: the analytic gradient matches finite differences

The closed-form gradient must agree with a numerical gradient of the loss. We confirm they match
to many digits.

In [ ]:
print(f'max |analytic - finite-diff| = {np.abs(grad - fd).max():.2e}')
assert np.abs(grad - fd).max() < 1e-6, 'the analytic gradient p_hat - one_hot matches finite differences'
print('\n✅ gradient-checking confirms the derived formula')

## Visualise: how loss varies with the correct-class logit

In [ ]:
z_sweep = np.linspace(-3, 5, 200)
losses = [cross_entropy_loss(np.array([z0, 1.0, -1.0]), 0) for z0 in z_sweep]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(z_sweep, losses, color='#6366f1', lw=2)
ax1.axvline(2.0, color='#f59e0b', ls='--', label='z₀ = 2 (worked example)')
ax1.set_xlabel('z₀ (correct-class logit)')
ax1.set_ylabel('Cross-entropy loss')
ax1.set_title('Loss vs correct-class logit')
ax1.legend(); ax1.grid(True, alpha=0.2)

# Gradient visualisation
grads_correct = [softmax_ce_grad(np.array([z0, 1.0, -1.0]), 0)[0] for z0 in z_sweep]
ax2.plot(z_sweep, grads_correct, color='#22d3ee', lw=2, label='∂L/∂z₀ = p̂₀ − 1')
ax2.axhline(0, color='#444', lw=0.8)
ax2.axvline(2.0, color='#f59e0b', ls='--', label='z₀ = 2')
ax2.set_xlabel('z₀')
ax2.set_ylabel('Gradient')
ax2.set_title('Gradient of correct-class logit')
ax2.legend(); ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.suptitle('Softmax + Cross-Entropy: loss and gradient landscape', y=1.02)
plt.show()

## Log-sum-exp numerical stability

In [ ]:
# Demonstrate overflow without the max-subtraction trick
z_large = np.array([1000.0, 999.0, 998.0])

print("Direct exp (no stability trick):")
print(f"  np.exp(1000) = {np.exp(1000.0)}")  # inf

print("\nWith max subtraction:")
e = np.exp(z_large - z_large.max())
p = e / e.sum()
print(f"  softmax({z_large}) = {p.round(4)}")
print(f"  sum = {p.sum():.6f}  (correct)")

### Validate: the max-subtraction trick prevents overflow

Exponentiating large logits directly overflows to `inf`. Subtracting the max first is
mathematically identical (softmax is shift-invariant) but numerically safe. We confirm the naive
overflow and the stable result.

In [ ]:
print(f'np.exp(1000) = {np.exp(1000.0)}')
assert np.isinf(np.exp(1000.0)), 'exp of a large logit overflows to inf without the trick'
assert np.isclose(p.sum(), 1.0), 'subtracting the max gives a finite softmax that sums to 1'
print('\n✅ subtract the max before exp — same math, no overflow')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **no max-subtraction** | exp overflows to inf (verified) |
| **separate softmax + log** | compute log-softmax directly for stability |
| **temperature** | mis-set T over-sharpens or over-flattens (demo) |
| **class imbalance** | plain CE favors majority classes — weight it |
| **label smoothing** | softens one-hot targets to curb overconfidence |

Demo: temperature sharpens (low T) or flattens (high T) the softmax.

In [ ]:
# TEMPERATURE scaling on the softmax: dividing logits by a temperature T reshapes the
# distribution. Low T (<1) SHARPENS it toward the argmax (more confident); high T (>1) FLATTENS it
# toward uniform (more uncertain). This is the knob behind sampling diversity and distillation.
# We confirm the sharpening/flattening.
softmax_temp = lambda zz, T: softmax(zz / T)
sharp = softmax_temp(z, 0.1)
flat = softmax_temp(z, 100.0)
print(f'max prob: T=0.1 (sharp) {sharp.max():.3f},  T=1 {softmax(z).max():.3f},  T=100 (flat) {flat.max():.3f}')
assert sharp.max() > softmax(z).max() > flat.max(), 'low T sharpens toward argmax; high T flattens toward uniform'
print('\nTemperature is the confidence dial: low = peaky/greedy, high = flat/exploratory.')

## ✏️ Your turn

**Exercise 1:** Extend `softmax_ce_grad` to handle a **batch** of examples. Input `Z` is `(N, K)`, `y` is `(N,)`. Return the mean gradient over the batch.

**Exercise 2:** Implement **temperature-scaled softmax** `softmax(z, T)` where dividing by temperature $T$ before softmax controls sharpness. Verify: at $T \to 0$ it approaches argmax; at $T \to \infty$ it approaches uniform.

**Exercise 3:** Show that for binary classification ($K=2$), softmax + cross-entropy is equivalent to sigmoid + binary cross-entropy. Verify numerically.

In [ ]:
# Exercise 1: batch gradient
def batch_softmax_ce_grad(Z, y):
    """
    Z: (N, K) logits
    y: (N,)  true class indices
    Returns: (K,) mean gradient
    """
    # TODO(you): vectorise the single-example formula
    pass

# Test
Z_test = np.array([[2.0, 1.0, -1.0], [0.5, 2.0, 1.0]])
y_test = np.array([0, 1])
# assert batch_softmax_ce_grad(Z_test, y_test).shape == (3,)

In [ ]:
# Exercise 2: temperature-scaled softmax
def softmax_temp(z, T=1.0):
    # TODO(you): divide logits by T before softmax
    pass

# for T in [0.01, 0.5, 1.0, 5.0, 100.0]:
#     print(f"T={T:6.2f}: {softmax_temp(z, T).round(3)}")

<details>
<summary>Solution — Exercise 1</summary>

```python
def batch_softmax_ce_grad(Z, y):
    N, K = Z.shape
    # Compute softmax for each row
    e = np.exp(Z - Z.max(axis=1, keepdims=True))
    P = e / e.sum(axis=1, keepdims=True)  # (N, K)
    # Subtract 1 from the true-class column
    P[np.arange(N), y] -= 1
    return P.mean(axis=0)  # (K,)
```
</details>

<details>
<summary>Solution — Exercise 2</summary>

```python
def softmax_temp(z, T=1.0):
    z_scaled = z / T
    e = np.exp(z_scaled - z_scaled.max())
    return e / e.sum()
```
</details>

## Key takeaways

- **Softmax + cross-entropy:** logits -> distribution -> $-\log$ of the true-class probability.
- **The gradient is $\hat p - \text{one-hot}(y)$** (verified against finite differences).
- **Numerical stability:** subtract the max before exp (verified) — otherwise overflow.
- **Temperature** sharpens (low T) or flattens (high T) the distribution (demo).